In [1]:
import os
import glob
from collections import defaultdict, Counter
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from pymutspec.annotation import CodonAnnotation, mutations_summary

from scipy.stats import chisquare, ks_2samp, pearsonr, spearmanr, uniform
from scipy.spatial.distance import cityblock
from sklearn.metrics import mean_squared_error
from Bio import SeqIO

import warnings
warnings.filterwarnings("ignore")

from aminoacidshift import (
    prepare_exp_aa_subst, prepare_aa_subst, get_equilibrium_freqs, 
    calc_metrics, amino_acid_codes
)


## Load input data

In [2]:
coda = CodonAnnotation(2)

# load spectra
spectra = pd.read_csv(
    '../../../192/1data_derivation/dataset/MutSpecVertebrates12.csv.gz', index_col=[0,2])

# load mutations
mut_df = pd.read_csv('../02_vertebrates/dataset/obs_muts.csv')\
    .rename(columns={'RefAa': 'aa1', 'AltAa': 'aa2', 'ProbaMut': 'count'})


In [3]:
spectra.head()

Class  Mut   Observed    Expected   MutSpec
Gene Species                                                        
CO1  Crocuta_crocuta  Mammalia  A>C        NaN  115.066667  0.000000
     Crocuta_crocuta  Mammalia  A>G  11.500444  173.333333  0.162818
     Crocuta_crocuta  Mammalia  A>T        NaN  115.066667  0.000000
     Crocuta_crocuta  Mammalia  C>A        NaN   71.466667  0.000000
     Crocuta_crocuta  Mammalia  C>G        NaN   71.466667  0.000000

In [4]:
cls_mean_spectra = spectra.groupby(['Gene', 'Class', 'Mut']).MutSpec.mean().reset_index(2)

In [5]:
mut_df.head()

,Mut,MutType,Site,PosInCodon,RefCodon,AltCodon,aa1,aa2,ProbaRef,count,RefNode,AltNode,species,gene,class,edge_nobs,species_nobs
0,A[A>G]T,0,112,1,ATT,GTT,I,V,0.958489,0.849141,Node5,Node6,Macaca_mulatta,ND6,Mammalia,1.70197,60.8728
1,T[G>A]T,0,127,1,GTT,ATT,V,I,0.974793,0.970650,Node6,Node12,Macaca_mulatta,ND6,Mammalia,4.31735,60.8728
2,G[A>G]T,0,301,1,ATA,GTA,M,V,0.999990,0.999990,Node24,RN_6,Macaca_mulatta,ND6,Mammalia,0.99999,60.8728
3,T[G>A]T,0,109,1,GTA,ATA,V,M,0.999990,0.999990,Node22,RN_32,Macaca_mulatta,ND6,Mammalia,0.99999,60.8728
4,A[A>G]T,0,97,1,ATT,GTT,I,V,1.000000,1.000000,Node25,RN_5,Macaca_mulatta,ND6,Mammalia,1.00000,60.8728


In [6]:
# load genes amino acid content

aa_content = []
for fasta_file in glob.glob('../../../192/1data_derivation/nemu_input/*.fasta'):
    file = SeqIO.parse(fasta_file, "fasta")
    gene = os.path.basename(fasta_file).replace('.fasta', '')
    for rec in file:
        seq = str(rec.seq)
        species = rec.description.split(';')[-1]
        species = '_'.join(species.split('_')[:-1])
        cur_aa_counts = dict(Counter(seq))
        cur_aa_counts['gene'] = gene
        cur_aa_counts['species'] = species
        aa_content.append(cur_aa_counts)
aa_content_df = pd.DataFrame(aa_content)\
    .set_index(['gene', 'species'])\
        .drop(['X', 'J', 'B', 'Z'], axis=1).fillna(0)\
            .rename(columns=amino_acid_codes)
aa_content_df


Lys   Asp   Ile   Gly   Thr  Leu   Tyr   Phe  \
gene species                                                                    
CO1  Abalistes_stellatus         2.0   8.0  21.0  22.0  14.0   36   5.0  15.0   
     Abantennarius_coccineus    11.0  14.0  40.0  48.0  36.0   62  18.0  42.0   
     Abbottina_rivularis        10.0  15.0  37.0  47.0  33.0   60  18.0  43.0   
     Ablennes_hians              8.0  15.0  41.0  47.0  36.0   61  18.0  42.0   
     Abramis_brama               8.0  15.0  38.0  46.0  34.0   62  18.0  42.0   
...                              ...   ...   ...   ...   ...  ...   ...   ...   
A8   Vireo_olivaceus             2.0   0.0   6.0   0.0   9.0    3   1.0   3.0   
     Xerotyphlops_vermicularis   3.0   0.0   4.0   0.0   9.0    6   2.0   1.0   
     Zosterops_flavifrons        2.0   0.0   3.0   0.0  11.0    6   1.0   3.0   
     Zosterops_griseotinctus     2.0   0.0   4.0   0.0   8.0    6   1.0   3.0   
     Zosterops_lateralis         2.0   0.0   3.0   0.0  11.0    6   1.0   3.0   

                                 Ala   Trp   Met   Val   Ser   Arg   Glu  Gln  \
gene species                                                                    
CO1  Abalistes_stellatus        24.0   6.0  11.0  12.0  13.0   3.0   2.0  5.0   
     Abantennarius_coccineus    43.0  17.0  30.0  39.0  30.0  10.0  12.0  6.0   
     Abbottina_rivularis        46.0  17.0  28.0  44.0  29.0   8.0  10.0  9.0   
     Ablennes_hians             46.0  17.0  24.0  42.0  29.0   8.0  11.0  9.0   
     Abramis_brama              49.0  17.0  25.0  43.0  29.0   8.0  11.0  8.0   
...                              ...   ...   ...   ...   ...   ...   ...  ...   
A8   Vireo_olivaceus             2.0   5.0   3.0   0.0   4.0   1.0   1.0  2.0   
     Xerotyphlops_vermicularis   1.0   3.0   5.0   1.0   3.0   0.0   0.0  1.0   
     Zosterops_flavifrons        0.0   5.0   2.0   1.0   6.0   0.0   0.0  2.0   
     Zosterops_griseotinctus     0.0   2.0   2.0   1.0   6.0   0.0   0.0  2.0   
     Zosterops_lateralis         0.0   5.0   2.0   1.0   6.0   0.0   0.0  2.0   

                                 Pro   Asn   His  Cys  
gene species                                           
CO1  Abalistes_stellatus        16.0   9.0   5.0  0.0  
     Abantennarius_coccineus    28.0  15.0  19.0  1.0  
     Abbottina_rivularis        28.0  16.0  17.0  1.0  
     Ablennes_hians             28.0  15.0  19.0  1.0  
     Abramis_brama              28.0  16.0  18.0  1.0  
...                              ...   ...   ...  ...  
A8   Vireo_olivaceus             9.0   4.0   0.0  0.0  
     Xerotyphlops_vermicularis   6.0   5.0   2.0  1.0  
     Zosterops_flavifrons        8.0   5.0   0.0  0.0  
     Zosterops_griseotinctus     7.0   5.0   0.0  0.0  
     Zosterops_lateralis         8.0   5.0   0.0  0.0  

[18937 rows x 20 columns]

## Explore mutations

In [7]:
mut_df.MutType.unique()

array([0])

In [8]:
# too low number of nonsyn mutations
mut_df.groupby(['gene', 'species'])['count'].sum().describe()

count    3904.000000
mean       17.582307
std        21.465450
min         5.001845
25%         7.729420
50%        10.997881
75%        18.856299
max       338.866032
Name: count, dtype: float64

## Fit model to distinct species

### Species-specific spectrum

In [9]:
metrics = []
# genes_set = set(['Cytb', 'ND2', 'CO1', 'CO3'])
genes_set = set(['Cytb', 'ND2'])
for (cur_gene, cur_species), obs in mut_df.groupby(['gene', 'species']):
    if cur_gene not in genes_set:
        continue
    
    if obs['count'].sum() < 25:
        continue
    
    try:
        ms12 = spectra.loc[(cur_gene, cur_species)]
    except KeyError:
        print((cur_gene, cur_species), "not found in spectra")
        continue

    try:
        cur_aa_content = aa_content_df.loc[(cur_gene, cur_species)]
    except KeyError:
        print((cur_gene, cur_species), "not found in aa_content_df")
        continue

    # calculate distance to equilibrium
    cur_eq_freq = get_equilibrium_freqs(ms12, gc=2)[1]
    cur_eq_freq['obs_cnt'] = cur_eq_freq['aa'].map(cur_aa_content.to_dict())
    cur_eq_freq['obs_freq'] = cur_eq_freq['obs_cnt'] / cur_eq_freq['obs_cnt'].sum()
    d = cityblock(cur_eq_freq['eq_freq'], cur_eq_freq['obs_freq'])

    ts_frac = ms12.query('Mut in ["C>T", "G>A", "T>C", "A>G"]').MutSpec.sum()
    tstv_ratio = ts_frac / (ms12.MutSpec.sum() - ts_frac)
    tstv_ratio = tstv_ratio if tstv_ratio < 100 else np.nan

    cur_cls = obs['class'].values[0]

    exp_aa_subst, _ = prepare_exp_aa_subst(ms12, 'MutSpec', 2)
    aa_subst = prepare_aa_subst(obs, exp_aa_subst, cur_aa_content.to_dict())
    cur_metrics = calc_metrics(aa_subst)
    cur_metrics['gene'] = cur_gene
    cur_metrics['class'] = cur_cls
    cur_metrics['species'] = cur_species
    cur_metrics['distance_to_eq'] = d
    cur_metrics['tstv_ratio'] = tstv_ratio
    metrics.append(cur_metrics)

metrics_df = pd.DataFrame(metrics).set_index(['gene', 'species'])
len(metrics_df)

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


...

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: 

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


469

In [10]:
metrics_df.sort_values('r2', ascending=False).head(10)

r2    r2_clr      mape      wape  \
gene species                                                               
Cytb Canis_lupus                  0.541420  0.539326  1.034783  0.622212   
     Craseonycteris_thonglongyai  0.508621  0.515158  0.486733  0.510872   
     Dipus_sagitta                0.431746  0.437723  0.615047  0.594223   
     Bos_taurus                   0.358384  0.356705  0.849552  0.657258   
     Cricetulus_barabensis        0.357730  0.354074  0.710178  0.658814   
     Rattus_rattus                0.340782  0.326653  0.661857  0.604355   
     Diploderma_polygonatum       0.331334  0.326613  0.722835  0.636626   
ND2  Anolis_pulchellus            0.323772  0.307946  0.736446  0.650168   
Cytb Ochotona_curzoniae           0.322553  0.314578  0.947973  0.661279   
ND2  Canis_lupus                  0.321816  0.320014  1.055547  0.700584   

                                  spearman_corr    spearman_p  \
gene species                                                    
Cytb Canis_lupus                       0.631144  1.935844e-12   
     Craseonycteris_thonglongyai       0.560387  1.469335e-06   
     Dipus_sagitta                     0.436336  4.535512e-07   
     Bos_taurus                        0.353085  6.182642e-05   
     Cricetulus_barabensis             0.388475  4.600417e-05   
     Rattus_rattus                     0.268430  3.881193e-03   
     Diploderma_polygonatum            0.478183  8.338292e-07   
ND2  Anolis_pulchellus                 0.307463  1.145343e-03   
Cytb Ochotona_curzoniae                0.380786  3.463984e-06   
ND2  Canis_lupus                       0.473391  3.192817e-06   

                                  pearson_corr_clr  pearson_p_clr  rmse_clr  \
gene species                                                                  
Cytb Canis_lupus                          0.735777   2.796422e-18  0.011315   
     Craseonycteris_thonglongyai          0.740319   2.730774e-12  0.022486   
     Dipus_sagitta                        0.674625   1.182208e-17  0.019607   
     Bos_taurus                           0.597269   3.055821e-13  0.015634   
     Cricetulus_barabensis                0.595124   2.695352e-11  0.019349   
     Rattus_rattus                        0.576081   1.998699e-11  0.018911   
     Diploderma_polygonatum               0.582980   4.601590e-10  0.017780   
ND2  Anolis_pulchellus                    0.565097   1.541539e-10  0.017381   
Cytb Ochotona_curzoniae                   0.564653   3.694354e-13  0.014014   
ND2  Canis_lupus                          0.567798   7.968922e-09  0.022303   

                                  log_likelihood  ...  nb_pearson_corr  \
gene species                                      ...                    
Cytb Canis_lupus                       -4.834618  ...         0.652015   
     Craseonycteris_thonglongyai       -4.277566  ...         0.677410   
     Dipus_sagitta                     -4.515132  ...         0.696022   
     Bos_taurus                        -4.807795  ...         0.567959   
     Cricetulus_barabensis             -4.259612  ...         0.641487   
     Rattus_rattus                     -5.230732  ...         0.598137   
     Diploderma_polygonatum            -4.799001  ...         0.587850   
ND2  Anolis_pulchellus                 -5.025566  ...         0.713275   
Cytb Ochotona_curzoniae                -5.569156  ...         0.416129   
ND2  Canis_lupus                       -4.772836  ...         0.394117   

                                  nb_pearson_p  nb_spearman_corr  \
gene species                                                       
Cytb Canis_lupus                  2.015754e-13          0.632080   
     Craseonycteris_thonglongyai  7.897011e-10          0.595407   
     Dipus_sagitta                4.035329e-19          0.440762   
     Bos_taurus                   7.332200e-12          0.452390   
     Cricetulus_barabensis        2.190085e-13          0.465951   
     Rattus_rattus              

In [11]:
ax=sns.boxplot(data=metrics_df, x='gene', y='r2', hue='class', palette='muted')
plt.grid(axis='y')
ax.set_axisbelow(True)
plt.ylabel('$R^2$', fontsize=12)
plt.xlabel('Ген', fontsize=12)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title='Класс')

In [12]:
metrics_df.groupby(['gene', 'class']).r2.describe()

count      mean       std       min       25%       50%  \
gene class                                                                   
Cytb Actinopteri    72.0 -0.006819  0.195315 -0.692413 -0.103531  0.016298   
     Amphibia       20.0  0.065806  0.100926 -0.096319 -0.005427  0.062634   
     Aves            9.0  0.073352  0.169152 -0.271295  0.044756  0.104432   
     Lepidosauria   40.0  0.044239  0.195518 -0.570497 -0.055629  0.107451   
     Mammalia      143.0  0.039805  0.255263 -1.440993 -0.043360  0.087285   
ND2  Actinopteri    18.0  0.127859  0.120191 -0.175122  0.086416  0.158518   
     Amphibia       25.0  0.109809  0.093789 -0.068845  0.061716  0.100500   
     Aves           43.0  0.033387  0.113034 -0.282587 -0.013297  0.040912   
     Lepidosauria   79.0  0.101781  0.152123 -0.464625  0.062944  0.126392   
     Mammalia       20.0  0.059633  0.164847 -0.326281  0.008574  0.090171   

                        75%       max  
gene class                             
Cytb Actinopteri   0.122202  0.303686  
     Amphibia      0.121528  0.292481  
     Aves          0.160371  0.295995  
     Lepidosauria  0.188157  0.331334  
     Mammalia      0.191481  0.541420  
ND2  Actinopteri   0.210021  0.312137  
     Amphibia      0.182662  0.296582  
     Aves          0.116084  0.191623  
     Lepidosauria  0.180842  0.323772  
     Mammalia      0.136539  0.321816

In [13]:
metrics_df.groupby(['gene']).mut_count.describe()

,count,mean,std,min,25%,50%,75%,max
gene,,,,,,,,
Cytb,284.0,47.146846,34.005873,11.969330,27.976205,35.684553,52.700246,338.866032
ND2,185.0,51.357699,43.977985,13.611276,28.937010,36.937024,54.875443,284.200121


In [14]:
ax=sns.boxplot(data=metrics_df, x='gene', y='mut_count', hue='class', palette='muted')
plt.grid(axis='y')
ax.set_axisbelow(True)
plt.ylabel('Количество мутаций', fontsize=12)
plt.xlabel('Ген', fontsize=12)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title='Класс')

In [15]:
metrics_df.tstv_ratio.hist(bins=25)

<Axes: xlabel='Ген', ylabel='Количество мутаций'>

In [16]:
cols = ['r2', 'mut_count', 'distance_to_eq', 'tstv_ratio']

# max_d = metrics_df['distance_to_eq'].quantile(0.99)
max_d = 10000
_ = metrics_df.query('distance_to_eq <= @max_d')[cols]

sns.heatmap(_.corr(), annot=True, cmap='rocket'
)

<Axes: >

In [17]:
sns.heatmap(metrics_df[cols].corr(method=lambda x, y: pearsonr(x, y)[1]), annot=True)

<Axes: >

In [18]:
sns.pairplot(metrics_df[cols].reset_index(), hue='gene', kind='reg', 
             plot_kws={'scatter_kws': {'alpha': 0.5, 's': 10}})

In [19]:
max_d = metrics_df['distance_to_eq'].quantile(0.99)

In [20]:
sns.lmplot(data=metrics_df.reset_index(), 
           x='r2', y='mut_count', col='gene', hue='class')

In [21]:
sns.lmplot(data=metrics_df.reset_index(), 
           x='r2', y='distance_to_eq', col='gene', hue='class')

In [22]:
sns.scatterplot(data=metrics_df, x='r2', y='distance_to_eq', hue='gene')
sns.regplot(data=metrics_df.query('gene == "Cytb"'), x='r2', y='distance_to_eq', color='b', scatter=False)
sns.regplot(data=metrics_df.query('gene == "ND2"'), x='r2', y='distance_to_eq', color='orange', scatter=False)

<Axes: title={'center': 'gene = ND2'}, xlabel='r2', ylabel='distance_to_eq'>

### Class-specific spectrum

In [23]:
metrics = []
# genes_set = set(['Cytb', 'ND2', 'CO1', 'CO3'])
genes_set = set(['Cytb', 'ND2'])
for (cur_gene, cur_species), obs in mut_df.groupby(['gene', 'species']):
    if cur_gene not in genes_set:
        continue
    
    if obs['count'].sum() < 25:
        continue
    
    try:
        cur_cls = obs['class'].values[0]
        ms12 = cls_mean_spectra.loc[(cur_gene, cur_cls)]
    except KeyError:
        print((cur_gene, cur_species), "not found in spectra")
        continue

    try:
        cur_aa_content = aa_content_df.loc[(cur_gene, cur_species)]
    except KeyError:
        print((cur_gene, cur_species), "not found in aa_content_df")
        continue

    # calculate distance to equilibrium
    cur_eq_freq = get_equilibrium_freqs(ms12, gc=2)[1]
    cur_eq_freq['obs_cnt'] = cur_eq_freq['aa'].map(cur_aa_content.to_dict())
    cur_eq_freq['obs_freq'] = cur_eq_freq['obs_cnt'] / cur_eq_freq['obs_cnt'].sum()
    d = cityblock(cur_eq_freq['eq_freq'], cur_eq_freq['obs_freq'])

    exp_aa_subst, _ = prepare_exp_aa_subst(ms12, 'MutSpec', 2)
    aa_subst = prepare_aa_subst(obs, exp_aa_subst, cur_aa_content.to_dict())
    cur_metrics = calc_metrics(aa_subst)
    cur_metrics['gene'] = cur_gene
    cur_metrics['class'] = cur_cls
    cur_metrics['species'] = cur_species
    cur_metrics['distance_to_eq'] = d
    metrics.append(cur_metrics)

metrics_df = pd.DataFrame(metrics).set_index(['gene', 'species'])
len(metrics_df)

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


Failed to calculate metrics for Zero-inflated Negative Binomial: Input contains NaN.


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


...

/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likel

Failed to calculate metrics for Zero-inflated Negative Binomial: Input contains NaN.


/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/home/kpotoh/AminoAcidShift/.venv/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


493

In [24]:
ax=sns.boxplot(data=metrics_df, x='gene', y='r2', hue='class', palette='muted')
plt.grid(axis='y')
ax.set_axisbelow(True)
plt.ylabel('$R^2$', fontsize=12)
plt.xlabel('Gene', fontsize=12)

Text(0.5, 28.888888888888857, 'Gene')

In [25]:
metrics_df.groupby(['gene', 'class']).r2.describe()

count      mean       std       min       25%       50%  \
gene class                                                                   
Cytb Actinopteri    76.0 -0.001728  0.214010 -1.069741 -0.101112  0.012819   
     Amphibia       20.0  0.076361  0.099012 -0.179569  0.040413  0.067480   
     Aves            9.0  0.094986  0.143118 -0.148493  0.054062  0.126368   
     Lepidosauria   40.0  0.060373  0.210094 -0.566507 -0.002280  0.120373   
     Mammalia      144.0  0.098260  0.166095 -0.333850  0.001593  0.125301   
ND2  Actinopteri    18.0  0.147112  0.099106 -0.028284  0.096334  0.190094   
     Amphibia       25.0  0.123170  0.110223 -0.159080  0.079027  0.128873   
     Aves           46.0  0.048172  0.128055 -0.451392  0.005084  0.055806   
     Lepidosauria   94.0  0.105386  0.147297 -0.531729  0.068626  0.129162   
     Mammalia       21.0  0.116059  0.130322 -0.211384  0.053858  0.110678   

                        75%       max  
gene class                             
Cytb Actinopteri   0.132736  0.319822  
     Amphibia      0.132269  0.268613  
     Aves          0.167852  0.291272  
     Lepidosauria  0.202302  0.344873  
     Mammalia      0.213907  0.575249  
ND2  Actinopteri   0.212360  0.321623  
     Amphibia      0.182733  0.353422  
     Aves          0.133299  0.223347  
     Lepidosauria  0.192645  0.352685  
     Mammalia      0.204896  0.351900

In [26]:
ax=sns.boxplot(data=metrics_df, x='gene', y='mut_count', hue='class', palette='muted')
plt.grid(axis='y')
ax.set_axisbelow(True)
plt.ylabel('Nonsyn Mutation Count', fontsize=12)
plt.xlabel('Gene', fontsize=12)

Text(0.5, 28.888888888888857, 'Gene')

In [27]:
cols = ['r2', 'mut_count', 'distance_to_eq']

max_d = metrics_df['distance_to_eq'].quantile(0.99)
# max_d = 10000
_ = metrics_df.query('distance_to_eq <= @max_d')[cols]

sns.heatmap(_.corr(), annot=True)

<Axes: title={'center': 'gene = ND2'}>

In [28]:
sns.heatmap(metrics_df[cols].corr(method=lambda x, y: pearsonr(x, y)[1])*3, annot=True)

<Axes: title={'center': 'gene = ND2'}>

In [29]:
sns.pairplot(metrics_df[cols].reset_index(), hue='gene', kind='reg', 
             plot_kws={'scatter_kws': {'alpha': 0.5, 's': 10}})

In [30]:
max_d = metrics_df['distance_to_eq'].quantile(0.99)

In [31]:
sns.scatterplot(data=metrics_df.query('distance_to_eq <= 100'), x='r2', y='distance_to_eq', hue='gene')
sns.regplot(data=metrics_df.query('distance_to_eq <= 100'), x='r2', y='distance_to_eq', color='black', scatter=False)

<Axes: xlabel='r2', ylabel='distance_to_eq'>